# Test Suite — CMAPSS Preprocessing & Scaler Validation

Generates and runs automated tests for all 4 CMAPSS test sets (FD001-FD004),
verifying:

1. Raw test files load with correct shape and no NaNs
2. Saved scalers (global + per-engine) load correctly
3. Applying saved scalers to test data produces values in [0,1]
4. No scaler is refit on test data (data leakage check)
5. Sliding window construction produces correct shapes
6. RUL-to-class label assignment is consistent with physics-based boundaries
7. Engine IDs in test data not seen during training fall back safely

Test files are written to `tests/` as both a pytest module and a
standalone notebook-runnable report.


In [1]:
import sys
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR= PROJECT_ROOT / "data" / "processed"
TESTS_DIR    = PROJECT_ROOT / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

FEATURE_COLS = [
    "op1","op2",
    "s2","s3","s4","s7","s8","s9",
    "s11","s12","s13","s14","s15","s17","s20","s21",
]
GLOBAL_SCALE_FEATURES = ["s3"]
PER_ENGINE_FEATURES   = [f for f in FEATURE_COLS if f not in GLOBAL_SCALE_FEATURES]

COLUMNS = (
    ["engine_id", "cycle"]
    + [f"op{i}" for i in range(1, 4)]
    + [f"s{i}"  for i in range(1, 22)]
)
DROP_COLS = ["s1","s5","s6","s10","s16","s18","s19","op3"]

SEQ_LEN = 30
RUL_CAP = 125

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw dir      : {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Tests dir    : {TESTS_DIR}")
print(f"Subsets      : {SUBSETS}")


Project root : C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine
Raw dir      : C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\data\raw
Processed dir: C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\data\processed
Tests dir    : C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\tests
Subsets      : ['FD001', 'FD002', 'FD003', 'FD004']


In [2]:
def load_cmapss(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.iloc[:, :len(COLUMNS)]
    df.columns = COLUMNS
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.drop(columns=DROP_COLS, errors="ignore")
    return df


def rul_to_class(rul: float) -> int:
    if rul > 100: return 0
    elif rul > 50: return 1
    elif rul > 10: return 2
    else: return 3


def apply_saved_scalers(df: pd.DataFrame, global_scaler, engine_scalers) -> pd.DataFrame:
    """
    Applies previously-fit scalers to test data WITHOUT refitting.
    This mirrors the production Phase 8/9/13 preprocessing path.
    Falls back to a fresh per-engine MinMaxScaler ONLY for engine_ids
    not present in engine_scalers (expected for unseen test engines).
    """
    df = df.copy()
    df[GLOBAL_SCALE_FEATURES] = global_scaler.transform(df[GLOBAL_SCALE_FEATURES])

    scaled_parts = []
    fallback_count = 0
    for engine_id, group in df.groupby("engine_id"):
        group = group.copy()
        if engine_id in engine_scalers:
            group[PER_ENGINE_FEATURES] = engine_scalers[engine_id].transform(group[PER_ENGINE_FEATURES])
        else:
            fallback_count += 1
            group[PER_ENGINE_FEATURES] = MinMaxScaler().fit_transform(group[PER_ENGINE_FEATURES])
        scaled_parts.append(group)

    result = pd.concat(scaled_parts).sort_values(["engine_id","cycle"]).reset_index(drop=True)
    return result, fallback_count


def get_last_window(df: pd.DataFrame, window_size: int = SEQ_LEN):
    X, engine_ids = [], []
    for engine_id, group in df.groupby("engine_id"):
        group = group.sort_values("cycle")
        feats = group[FEATURE_COLS].values
        if len(feats) >= window_size:
            window = feats[-window_size:]
        else:
            pad    = np.zeros((window_size - len(feats), feats.shape[1]))
            window = np.vstack([pad, feats])
        X.append(window)
        engine_ids.append(engine_id)
    return np.array(X, dtype=np.float32), engine_ids


print("Helper functions defined (mirrors production preprocessing).")


Helper functions defined (mirrors production preprocessing).


In [3]:
class TestResult:
    def __init__(self, name, passed, message=""):
        self.name = name
        self.passed = passed
        self.message = message

    def __repr__(self):
        status = "PASS" if self.passed else "FAIL"
        return f"[{status}] {self.name}" + (f" - {self.message}" if self.message else "")


all_results = []

def record(name, condition, message=""):
    result = TestResult(name, bool(condition), message)
    all_results.append(result)
    print(result)
    return result.passed


print("Test runner ready.")


Test runner ready.


In [4]:
subset_artifacts = {}

for subset in SUBSETS:
    print(f"\n{'='*60}")
    print(f"  TESTING {subset}")
    print(f"{'='*60}")

    test_path = RAW_DIR / f"test_{subset}.txt"
    rul_path  = RAW_DIR / f"RUL_{subset}.txt"
    gs_path   = PROCESSED_DIR / subset / "global_scaler.pkl"
    es_path   = PROCESSED_DIR / subset / "engine_scalers.pkl"

    # ---- Test 1: raw files exist ----
    record(f"{subset}: test file exists", test_path.exists(), str(test_path))
    record(f"{subset}: RUL file exists",  rul_path.exists(),  str(rul_path))

    if not (test_path.exists() and rul_path.exists()):
        print(f"  Skipping remaining {subset} tests - raw files missing.")
        continue

    # ---- Test 2: raw data loads correctly ----
    test_raw = load_cmapss(test_path)
    record(f"{subset}: test data loads", len(test_raw) > 0, f"{len(test_raw)} rows")
    record(f"{subset}: no NaNs in raw test data", test_raw.isnull().sum().sum() == 0)
    record(f"{subset}: has all FEATURE_COLS",
           all(c in test_raw.columns for c in FEATURE_COLS),
           f"missing: {[c for c in FEATURE_COLS if c not in test_raw.columns]}")

    # ---- Test 3: RUL file loads correctly ----
    rul_df = pd.read_csv(rul_path, header=None, names=["RUL"])
    n_engines_test = test_raw["engine_id"].nunique()
    record(f"{subset}: RUL count matches engine count",
           len(rul_df) == n_engines_test,
           f"RUL rows={len(rul_df)}, engines={n_engines_test}")

    # ---- Test 4: scalers exist and load ----
    scalers_exist = gs_path.exists() and es_path.exists()
    record(f"{subset}: global_scaler.pkl exists", gs_path.exists())
    record(f"{subset}: engine_scalers.pkl exists", es_path.exists())

    if not scalers_exist:
        print(f"  Skipping scaler-dependent tests for {subset} - scalers missing.")
        continue

    global_scaler  = joblib.load(gs_path)
    engine_scalers = joblib.load(es_path)
    record(f"{subset}: global_scaler is MinMaxScaler instance",
           isinstance(global_scaler, MinMaxScaler))
    record(f"{subset}: engine_scalers is non-empty dict",
           isinstance(engine_scalers, dict) and len(engine_scalers) > 0,
           f"{len(engine_scalers)} engines")

    # ---- Test 5: apply scalers without refitting (leakage check) ----
    test_scaled, fallback_count = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
    record(f"{subset}: scaled output same row count as input",
           len(test_scaled) == len(test_raw))
    print(f"  Engines using fallback scaler (unseen during training): {fallback_count}")

    # ---- Test 6: scaled values in valid range ----
    feat_min = test_scaled[FEATURE_COLS].min().min()
    feat_max = test_scaled[FEATURE_COLS].max().max()
    record(f"{subset}: scaled features within [-0.05, 1.05] tolerance",
           feat_min >= -0.05 and feat_max <= 1.05,
           f"min={feat_min:.4f}, max={feat_max:.4f}")

    # ---- Test 7: sliding window construction ----
    X_test, engine_ids = get_last_window(test_scaled)
    record(f"{subset}: window shape correct",
           X_test.shape == (n_engines_test, SEQ_LEN, len(FEATURE_COLS)),
           f"got {X_test.shape}")
    record(f"{subset}: no NaNs in windowed data", not np.isnan(X_test).any())

    # ---- Test 8: RUL to class label consistency ----
    y_rul    = rul_df["RUL"].values
    y_class  = np.array([rul_to_class(r) for r in y_rul])
    record(f"{subset}: all class labels in {{0,1,2,3}}",
           set(np.unique(y_class)).issubset({0,1,2,3}))
    record(f"{subset}: class boundaries match RUL thresholds",
           all(
               (y_class[i] == 0 and y_rul[i] > 100) or
               (y_class[i] == 1 and 50 < y_rul[i] <= 100) or
               (y_class[i] == 2 and 10 < y_rul[i] <= 50) or
               (y_class[i] == 3 and y_rul[i] <= 10)
               for i in range(len(y_rul))
           ))

    subset_artifacts[subset] = {
        "X_test": X_test,
        "y_rul": y_rul,
        "y_class": y_class,
        "engine_ids": engine_ids,
        "fallback_count": fallback_count,
        "n_engines": n_engines_test,
    }

print(f"\n{'='*60}")
print("All per-subset tests complete.")
print(f"{'='*60}")



  TESTING FD001
[PASS] FD001: test file exists - C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\data\raw\test_FD001.txt
[PASS] FD001: RUL file exists - C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\data\raw\RUL_FD001.txt
[PASS] FD001: test data loads - 13096 rows
[PASS] FD001: no NaNs in raw test data
[PASS] FD001: has all FEATURE_COLS - missing: []
[PASS] FD001: RUL count matches engine count - RUL rows=100, engines=100
[PASS] FD001: global_scaler.pkl exists
[PASS] FD001: engine_scalers.pkl exists
[PASS] FD001: global_scaler is MinMaxScaler instance
[PASS] FD001: engine_scalers is non-empty dict - 100 engines
[PASS] FD001: scaled output same row count as input
  Engines using fallback scaler (unseen during training): 0
[FAIL] FD001: scaled features within [-0.05, 1.05] tolerance - min=-1.3261, max=5.3354
[PASS] FD001: window shape correct - got (100, 30, 16)
[PASS] FD001: no NaNs in windowed data
[PASS] FD001: all class labels in {0,1,2,3}
[PASS] FD001:

In [5]:
print("CROSS-SUBSET CONSISTENCY TESTS")
print("=" * 60)

# Test: feature dimension consistent across all subsets
input_dims = set()
for subset, art in subset_artifacts.items():
    input_dims.add(art["X_test"].shape[2])
record("All subsets have same INPUT_DIM (16)",
       input_dims == {16}, f"found dims: {input_dims}")

# Test: no subset has zero test engines
for subset, art in subset_artifacts.items():
    record(f"{subset}: has at least 1 test engine", art["n_engines"] > 0)

# Test: class distribution sanity - C0 should generally be largest or near-largest
for subset, art in subset_artifacts.items():
    from collections import Counter
    dist = Counter(art["y_class"].tolist())
    record(f"{subset}: all 4 classes present in test set or gracefully empty",
           True,  # informational - just log distribution
           f"distribution: {dict(sorted(dist.items()))}")

print("=" * 60)


CROSS-SUBSET CONSISTENCY TESTS
[PASS] All subsets have same INPUT_DIM (16) - found dims: {16}
[PASS] FD001: has at least 1 test engine
[PASS] FD002: has at least 1 test engine
[PASS] FD003: has at least 1 test engine
[PASS] FD004: has at least 1 test engine
[PASS] FD001: all 4 classes present in test set or gracefully empty - distribution: {0: 33, 1: 34, 2: 26, 3: 7}
[PASS] FD002: all 4 classes present in test set or gracefully empty - distribution: {0: 92, 1: 79, 2: 66, 3: 22}
[PASS] FD003: all 4 classes present in test set or gracefully empty - distribution: {0: 32, 1: 39, 2: 23, 3: 6}
[PASS] FD004: all 4 classes present in test set or gracefully empty - distribution: {0: 100, 1: 68, 2: 66, 3: 14}


In [6]:
print("SCALER-SPECIFIC SANITY TESTS")
print("=" * 60)

s3_idx = FEATURE_COLS.index("s3")
for subset, art in subset_artifacts.items():
    s3_std = art["X_test"][:, :, s3_idx].std()
    record(f"{subset}: s3 has non-zero variance after global scaling",
           s3_std > 0.001, f"std={s3_std:.4f}")

# verify op3 truly absent (dropped permanently, not just zero)
for subset in SUBSETS:
    test_path = RAW_DIR / f"test_{subset}.txt"
    if not test_path.exists():
        continue
    raw_check = load_cmapss(test_path)
    record(f"{subset}: op3 absent from processed columns",
           "op3" not in raw_check.columns)

print("=" * 60)


SCALER-SPECIFIC SANITY TESTS
[PASS] FD001: s3 has non-zero variance after global scaling - std=0.1161
[PASS] FD002: s3 has non-zero variance after global scaling - std=0.2892
[PASS] FD003: s3 has non-zero variance after global scaling - std=0.1137
[PASS] FD004: s3 has non-zero variance after global scaling - std=0.2868
[PASS] FD001: op3 absent from processed columns
[PASS] FD002: op3 absent from processed columns
[PASS] FD003: op3 absent from processed columns
[PASS] FD004: op3 absent from processed columns


In [7]:
n_pass = sum(1 for r in all_results if r.passed)
n_fail = sum(1 for r in all_results if not r.passed)
n_total = len(all_results)

print("=" * 60)
print("           TEST SUITE SUMMARY")
print("=" * 60)
print(f"Total tests : {n_total}")
print(f"Passed      : {n_pass}")
print(f"Failed      : {n_fail}")
print(f"Pass rate   : {n_pass/n_total*100:.1f}%")
print("=" * 60)

if n_fail > 0:
    print("\nFAILED TESTS:")
    for r in all_results:
        if not r.passed:
            print(f"  - {r.name}: {r.message}")

# save report
report = {
    "total": n_total,
    "passed": n_pass,
    "failed": n_fail,
    "pass_rate_pct": round(n_pass/n_total*100, 2),
    "results": [
        {"name": r.name, "passed": r.passed, "message": r.message}
        for r in all_results
    ],
}

with open(TESTS_DIR / "test_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"\nSaved test_report.json -> {TESTS_DIR}")


           TEST SUITE SUMMARY
Total tests : 81
Passed      : 77
Failed      : 4
Pass rate   : 95.1%

FAILED TESTS:
  - FD001: scaled features within [-0.05, 1.05] tolerance: min=-1.3261, max=5.3354
  - FD002: scaled features within [-0.05, 1.05] tolerance: min=-0.0635, max=1.2982
  - FD003: scaled features within [-0.05, 1.05] tolerance: min=-5.3810, max=5.7132
  - FD004: scaled features within [-0.05, 1.05] tolerance: min=-0.0701, max=1.2149

Saved test_report.json -> C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\tests


In [8]:
pytest_code = '''"""
Automated pytest suite for CMAPSS preprocessing and scaler validation.
Auto-generated from tests/generate_tests.ipynb - do not edit results by hand,
rerun the notebook to regenerate this file if the pipeline changes.

Run with: pytest tests/test_preprocessing.py -v
"""

import json
import numpy as np
import pandas as pd
import joblib
import pytest
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path(__file__).parent.parent
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

FEATURE_COLS = [
    "op1","op2",
    "s2","s3","s4","s7","s8","s9",
    "s11","s12","s13","s14","s15","s17","s20","s21",
]
GLOBAL_SCALE_FEATURES = ["s3"]
PER_ENGINE_FEATURES   = [f for f in FEATURE_COLS if f not in GLOBAL_SCALE_FEATURES]

COLUMNS = (
    ["engine_id", "cycle"]
    + [f"op{i}" for i in range(1, 4)]
    + [f"s{i}" for i in range(1, 22)]
)
DROP_COLS = ["s1","s5","s6","s10","s16","s18","s19","op3"]
SEQ_LEN = 30


def load_cmapss(path):
    df = pd.read_csv(path, sep=r"\\s+", header=None)
    df = df.iloc[:, :len(COLUMNS)]
    df.columns = COLUMNS
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.drop(columns=DROP_COLS, errors="ignore")
    return df


def rul_to_class(rul):
    if rul > 100: return 0
    elif rul > 50: return 1
    elif rul > 10: return 2
    else: return 3


def apply_saved_scalers(df, global_scaler, engine_scalers):
    df = df.copy()
    df[GLOBAL_SCALE_FEATURES] = global_scaler.transform(df[GLOBAL_SCALE_FEATURES])
    scaled_parts = []
    for engine_id, group in df.groupby("engine_id"):
        group = group.copy()
        if engine_id in engine_scalers:
            group[PER_ENGINE_FEATURES] = engine_scalers[engine_id].transform(group[PER_ENGINE_FEATURES])
        else:
            group[PER_ENGINE_FEATURES] = MinMaxScaler().fit_transform(group[PER_ENGINE_FEATURES])
        scaled_parts.append(group)
    return pd.concat(scaled_parts).sort_values(["engine_id","cycle"]).reset_index(drop=True)


def get_last_window(df, window_size=SEQ_LEN):
    X = []
    for engine_id, group in df.groupby("engine_id"):
        group = group.sort_values("cycle")
        feats = group[FEATURE_COLS].values
        if len(feats) >= window_size:
            window = feats[-window_size:]
        else:
            pad = np.zeros((window_size - len(feats), feats.shape[1]))
            window = np.vstack([pad, feats])
        X.append(window)
    return np.array(X, dtype=np.float32)


@pytest.fixture(params=SUBSETS)
def subset(request):
    return request.param


@pytest.fixture
def test_raw(subset):
    path = RAW_DIR / f"test_{subset}.txt"
    if not path.exists():
        pytest.skip(f"test_{subset}.txt not found")
    return load_cmapss(path)


@pytest.fixture
def rul_df(subset):
    path = RAW_DIR / f"RUL_{subset}.txt"
    if not path.exists():
        pytest.skip(f"RUL_{subset}.txt not found")
    return pd.read_csv(path, header=None, names=["RUL"])


@pytest.fixture
def scalers(subset):
    gs_path = PROCESSED_DIR / subset / "global_scaler.pkl"
    es_path = PROCESSED_DIR / subset / "engine_scalers.pkl"
    if not (gs_path.exists() and es_path.exists()):
        pytest.skip(f"scalers for {subset} not found")
    return joblib.load(gs_path), joblib.load(es_path)


class TestRawDataIntegrity:
    def test_no_nans(self, test_raw):
        assert test_raw.isnull().sum().sum() == 0

    def test_has_feature_columns(self, test_raw):
        missing = [c for c in FEATURE_COLS if c not in test_raw.columns]
        assert not missing, f"missing columns: {missing}"

    def test_op3_dropped(self, test_raw):
        assert "op3" not in test_raw.columns

    def test_rul_count_matches_engines(self, test_raw, rul_df):
        assert len(rul_df) == test_raw["engine_id"].nunique()


class TestScalerApplication:
    def test_global_scaler_is_minmax(self, scalers):
        global_scaler, _ = scalers
        assert isinstance(global_scaler, MinMaxScaler)

    def test_engine_scalers_nonempty(self, scalers):
        _, engine_scalers = scalers
        assert isinstance(engine_scalers, dict) and len(engine_scalers) > 0

    def test_scaled_values_in_range(self, test_raw, scalers):
        global_scaler, engine_scalers = scalers
        scaled = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
        assert scaled[FEATURE_COLS].min().min() >= -0.05
        assert scaled[FEATURE_COLS].max().max() <= 1.05

    def test_no_row_loss_after_scaling(self, test_raw, scalers):
        global_scaler, engine_scalers = scalers
        scaled = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
        assert len(scaled) == len(test_raw)

    def test_s3_has_variance_after_global_scaling(self, test_raw, scalers):
        global_scaler, engine_scalers = scalers
        scaled = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
        assert scaled["s3"].std() > 0.001


class TestWindowConstruction:
    def test_window_shape(self, test_raw, scalers):
        global_scaler, engine_scalers = scalers
        scaled = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
        X = get_last_window(scaled)
        n_engines = test_raw["engine_id"].nunique()
        assert X.shape == (n_engines, SEQ_LEN, len(FEATURE_COLS))

    def test_no_nans_in_windows(self, test_raw, scalers):
        global_scaler, engine_scalers = scalers
        scaled = apply_saved_scalers(test_raw, global_scaler, engine_scalers)
        X = get_last_window(scaled)
        assert not np.isnan(X).any()


class TestLabelConsistency:
    def test_rul_to_class_boundaries(self, rul_df):
        for rul in rul_df["RUL"].values:
            c = rul_to_class(rul)
            assert c in {0, 1, 2, 3}
            if c == 0: assert rul > 100
            if c == 1: assert 50 < rul <= 100
            if c == 2: assert 10 < rul <= 50
            if c == 3: assert rul <= 10
'''

with open(TESTS_DIR / "test_preprocessing.py", "w") as f:
    f.write(pytest_code)

print(f"Saved test_preprocessing.py -> {TESTS_DIR}")
print("\nRun with: pytest tests/test_preprocessing.py -v")


Saved test_preprocessing.py -> C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\tests

Run with: pytest tests/test_preprocessing.py -v


In [9]:
conftest_code = '''"""
Pytest configuration - ensures tests/ is discoverable as a package root
relative to the project, so imports resolve consistently regardless of
the directory pytest is invoked from.
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent))
'''

with open(TESTS_DIR / "conftest.py", "w") as f:
    f.write(conftest_code)

print(f"Saved conftest.py -> {TESTS_DIR}")
print("\nFinal tests/ directory contents:")
for f in sorted(TESTS_DIR.iterdir()):
    print(f"  {f.name}")


Saved conftest.py -> C:\Users\Ishaan Nandoskar\Downloads\nasa_gan_turbo_fan_engine\tests

Final tests/ directory contents:
  conftest.py
  generate_tests.ipynb
  test_preprocessing.py
  test_report.json
